In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from reprojection import *
from utils import *
from interpolation import *


In [2]:
files = sorted(glob.glob('/home/ulyanov/data/cross/*'))
files

['/home/ulyanov/data/cross/hmi.M_45s.20241120_001500_TAI.2.magnetogram.fits',
 '/home/ulyanov/data/cross/hmi.M_720s.20241120_001200_TAI.3.magnetogram.fits',
 '/home/ulyanov/data/cross/hmi.V_45s.20241120_001500_TAI.2.Dopplergram.fits',
 '/home/ulyanov/data/cross/hmi.V_720s.20241120_001200_TAI.3.Dopplergram.fits',
 '/home/ulyanov/data/cross/solo_L2_phi-fdt-blos_20241120T001503_V202602220151_0451200501.fits.gz',
 '/home/ulyanov/data/cross/solo_L2_phi-fdt-vlos_20241120T001503_V202602220151_0451200501.fits.gz']

In [3]:
s = np.load('/home/ulyanov/data/solo/phi/distortion/fdt/distortion.npz')
xd, yd = s['xd'], s['yd']

In [4]:
file_hmi = files[1]
file_fdt = files[4]

with fits.open(file_hmi) as hdul:
    header_hmi = hdul[1].header
    data_hmi = hdul[1].data

with fits.open(file_fdt) as hdul:
    header_fdt = hdul[0].header
    data_fdt = hdul[0].data

data_fdt = undistort(data_fdt, header_fdt, xd, yd)

data_hmi = rebin(data_hmi, 8, update_header=header_hmi)
data_hmi = reproject(data_hmi, header_hmi, header_fdt, correct_mu=False, mu_thr=0.1, crota=header_fdt['CROTA'] + 0.29, rsun=330.25 + 0.28, xc=421.64, yc=457.87)

data_fdt[np.isnan(data_hmi)] = np.nan

In [5]:
ft_fdt = np.fft.fft2(np.nan_to_num(data_fdt))
ft_hmi = np.fft.fft2(np.nan_to_num(data_hmi))

#ft_psf = ft_fdt * np.conj(ft_hmi) / (np.abs(ft_hmi) ** 2 + 1e-4)
ft_psf = ft_hmi * np.conj(ft_fdt) / (np.abs(ft_fdt) ** 2 + 1e-4)
psf = np.real(np.fft.ifft2(ft_psf))
psf = np.roll(psf, (4, 4), axis=(0,1))
psf = psf[:9,:9]

plt.figure(figsize=(10,10))
plt.imshow(psf, cmap='gray')

In [6]:
from scipy.signal import convolve2d

data_fdt = convolve2d(np.nan_to_num(data_fdt), psf, mode='same')
data_fdt[np.isnan(data_fdt)] = np.nan

plt.figure(figsize=(10,10))
plt.imshow(data_fdt, cmap='seismic', vmin=-50, vmax=50)
plt.tight_layout()

In [5]:
xx, yy = [0], [0]

for q in np.arange(2., 30, 0.5):
    w = np.sqrt(data_hmi ** 2 + data_fdt ** 2)
    t = np.where(w < q ** 2)
    x, y = data_hmi[t], data_fdt[t]
    w = w[t] ** 3

    A = np.array([[np.mean(w * x ** 2), np.mean(w * x * y)],
                  [np.mean(w * x * y), np.mean(w * y ** 2)]])

    vals, vecs = np.linalg.eigh(A)
    u, v = vecs[:, 1]
    u, v = u * q ** 2 * np.sign(u), v * q ** 2 * np.sign(u)

    xx = [-u] + xx + [u]
    yy = [-v] + yy + [v]

xx = np.array(xx)
yy = np.array(yy)

In [6]:
plt.figure(figsize=(10,10))
plt.scatter(data_hmi, data_fdt, s=0.05)
plt.plot(xx, yy, '--', color='black')

plt.xlabel('HMI, G')
plt.ylabel('FDT, G')

#plt.xlim(-40,40)
#plt.ylim(-40,40)

plt.xlim(-200,200)
plt.ylim(-200,200)

plt.grid(True)
plt.tight_layout()

In [ ]:
np.savez('cross_calibration.npz', hmi=xx, fdt=yy, psf=psf)

In [ ]:
data_fdt_ = hmize(data_fdt, fdt=yy, hmi=xx)

In [ ]:
plt.figure(figsize=(10,10))
plt.scatter(data_hmi, data_fdt_, s=0.01)

plt.xlim(-500,500)
plt.ylim(-500,500)

In [5]:
plt.figure(figsize=(10,10))
plt.imshow(data_hmi, cmap='seismic', vmin=-50, vmax=50)
plt.tight_layout()

In [6]:
plt.figure(figsize=(10,10))
plt.imshow(data_fdt, cmap='seismic', vmin=-50, vmax=50)
plt.tight_layout()

In [11]:
np.arctan2(1, 0)

np.float64(1.5707963267948966)